# MXN441: Assignment 2 Code

**Authors:**  
Jack Fewtrell (n11278331)   
Madeline Miller (n10267646)   

**Selected Paper:**    
*Hybrid ViT-CNN Network for Fine-Grained Image Classification*   

**Description:**  
This notebook contains code developed for Assignment 2 in MXN441, based on the selected paper above.

## Required packages 


In [1]:
import numpy as np
import pandas as pd
import re

import torch
import torch.nn as nn
import torch.nn.functional as F

# Pillow for images 
from PIL import Image

# Test train split 
from sklearn.model_selection import train_test_split

## Importing Data 

The version of the dataset can be found on Kaggle, due to the size of the data set, it cannot be included in the submission. A version can be found through the following link on OneDrive: 

The dataset is imported by following the process below. 

In [4]:
# Accessing the classes 
nclass = 200 # number of classes 
classes = []

# Path to the folder containing the CUB-200-2011 dataset
data_location = "Data/CUB_200_2011"
# data_location = r"C:\Users\n10267646\Downloads\CUB_200_2011\CUB_200_2011" # *** TO BE REMOVED BEFORE SUBMISSION


with open(f"{data_location}/classes.txt") as f:
  for x in f:
    match = re.search(r"\d+\.[A-Za-z_]+", x)
    if match:
        classes.append(match.group(0))

# Accessing the image names and ID's 
image_ids = []
image_names = []
with open(f"{data_location}/images.txt") as f:
  for x in f:
    match = re.search(r"(\d+)\s+(.+)", x)
    if match:
        image_ids.append(match.group(1))
        image_names.append(match.group(2))

# Getting image class labels 
image_classes = []
with open(f"{data_location}/image_class_labels.txt") as f:
    for x in f: 
        match = re.search(r" (\d+)", x)
        if match:
            temp = (match.group(1))
            image_classes.append(classes[int(temp) - 1])

# import the path to the images 
image_path = []
for x in range(len(image_names)):
    image_path.append(f"{data_location}/images/{image_names[x]}")

# saving images to memory takes time, may open them as needed 
#for x in range(len(image_names)):
#    img = Image.open(f"{data_location}/images/{image_names[x]}") 
#    images.append(img.copy())
#    img.close()

# importing provided training and testing sets 
isTraining = [] 
with open(f"{data_location}/train_test_split.txt") as f:
    for x in f: 
        match = re.search(r"(\d+) (\d+)", x) 
        if match:
            isTraining.append(int(match.group(2)))

# creating a data frame, listing Images and their subsequent locations 
data = {
    "ID": image_ids,
    "Name": image_names,
    "Class": image_classes,
    "Image Path": image_path,
    "isTraining": isTraining
}

df = pd.DataFrame(data)


## Data Preprocessing 

The CUB dataset provides coordinates specifying the location of each bird within an image. The code below loads these coordinates and stores the associated boundary box information.

In [6]:
# Initialise dictionary to store bounding box coordinates for each image
bbox_dict = {}

with open(f"{data_location}/bounding_boxes.txt") as f:
    for line in f:
        # Split each line into image ID and boundary box values
        parts = line.strip().split()
        # Extract image ID
        img_id = int(parts[0])  
        # Extract boundary box coordinates
        x, y, w, h = map(float, parts[1:]) # (x,y) = position of top-left corner, (w,h) = width and height of boundary box in pixels
        # Store bounding box information using image ID as the key
        bbox_dict[img_id] = (x, y, w, h)   

The function below is used for image preprocessing. Each image is cropped to its bounding box with additional padding applied around the border. The cropped image is then resized to the specified dimensions. 

In [8]:
# Defining image preprocessing pipeline 
def getImage(path, size, img_ID, bbox_dict, pad = 0.15):
    """
    Image preprocessing pipeline
    Loads an image from path, applies bounding box cropping with padding, and resizes the image for use in the ML models.

    Input:
    path (str): path to image.
    size (int): output image size.
    img_ID (int): image ID used to access boundary box information in bbox_dict.
    bbox_dict (dict): dictionary containing boundary box information.
    pad (float): % padding added around bounding box.

    Output: 
    image: PIL.Image.
    
    """
    # Open image 
    img = Image.open(path)
    # Copy a local instance
    image = img.copy()
    # Close original image 
    img.close()

    # Get bounding box information
    x, y, w, h = bbox_dict[img_ID]

    # Get image dimensions
    img_w, img_h = image.size
    
    # Compute padding to add to boundary box (in pixels)
    pad_w = w * pad
    pad_h = h * pad

    # Define padded crop coordinates
    x1 = max(0, int(x - pad_w))
    y1 = max(0, int(y - pad_h))
    x2 = min(img_w, int(x + w + pad_w))
    y2 = min(img_h, int(y + h + pad_h))

    # Crop image using bounding box
    image = image.crop((x1, y1, x2, y2))

    # Resize cropped image
    image = image.resize((size, size))

    # Return processed image 
    return(image)

## Data Split 

The data set provided has it's own suggested training and test split. Therefore, this split will be kept. An additional validation set will be split from the training set for training purposes. This will be 30% of the training set. 

In [4]:
seed = 10 # seed for training split 

# Defining test set 
filtered_df = df[(df["isTraining"] == 0)]
testID = filtered_df["ID"]

# determining training - validation set 
filtered_df = df[(df["isTraining"] == 1)]
trainID, valID = train_test_split(
    filtered_df["ID"],
    test_size = 0.2,
    stratify = filtered_df["Class"],
    random_state = seed
)

## Paper Method Replication

**Selective Kernel (SK) Module**   

(Required for MIT module)

In [5]:
class SKUnit(nn.Module):
    """
    Selective Kernel (SK) Unit.

    Applies multi-scale convolution (kernel sizes: 3 and 5) and learns channel-wise attention to adaptively fuse them.

    Args:
        D (int): Number of input channels.
        r (int): Reduction ratio for bottleneck.

    Input:
        x (Tensor): Shape (B, D, H, W)

    Output:
        V (Tensor): Shape (B, D, H, W)
    """
    
    def __init__(self, D=32, r=4):

        super().__init__()
        
        # Store hyperparameters
        self.D = D
        self.r = r

        # --- Split: two convolution branches ---
        # Branch 1: 3*3
        self.conv3 = nn.Sequential(
            nn.Conv2d(in_channels=D, out_channels=D, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(D),
            nn.ReLU(inplace=True)
        )
        
        # Branch 2: 5*5
        self.conv5 = nn.Sequential(
            nn.Conv2d(in_channels=D, out_channels=D, kernel_size=5, padding=2, bias=False),
            nn.BatchNorm2d(D),
            nn.ReLU(inplace=True)
        )

        # --- Fuse: channel reduction  ---
        d = D // r # reduced channel dimension
        self.fc1 = nn.Linear(in_features=D, out_features=d, bias=False) 
        self.bn = nn.BatchNorm1d(d)            
        self.relu = nn.ReLU(inplace=True)   

        # --- Select: compute attention weights ---
        self.fc2 = nn.Linear(in_features=d, out_features=2*D, bias=False) 

    def forward(self, x):
        B, D, H, W = x.shape  

        # --- Split ---
        U1 = self.conv3(x)          # (B, D, H, W)
        U2 = self.conv5(x)          # (B, D, H, W)

        # --- Fuse ---
        # combine branches
        U = U1 + U2 
        
        # global average pooling
        s = U.mean(dim=(2, 3))      # (B, D)
        
        # channel reduction
        z = self.fc1(s)             # (B, d)
        z = self.bn(z)
        z = self.relu(z)

        # --- Select ---
        # compute attention
        attn = self.fc2(z)              # (B, 2D)
        attn = attn.view(B, 2, D)       # (B, 2, D)
        attn = F.softmax(attn, dim=1)   # across branches

        # extract branch weights
        a = attn[:, 0].unsqueeze(-1).unsqueeze(-1)  # (B, D, 1, 1)
        b = attn[:, 1].unsqueeze(-1).unsqueeze(-1)

        # weighted aggregation
        V = a * U1 + b * U2   # (B, D, H, W)
        
        return V

**Multi-scale Image-to-Tokens (MIT) module** 

In [6]:
class MIT(nn.Module):
    """
    Multi-scale Image-to-Tokens (MIT) module.

    Converts an image into a sequence of tokens using CNN feature extraction + patch embedding.

    Args:
        d (int): Token embedding dimension.
        D (int): CNN feature channels.
        P (int): Patch size.

    Input:
        x (Tensor): Shape (B, 3, H, W)

    Output:
        x (Tensor): Shape (B, N+1, d)
    """
    
    def __init__(self, d=384, D=32, P=4):

        super().__init__()

        # Store hyperparameters
        self.P = P 
        self.d = d  
        self.D = D 

        # --- Initial convolution + downsampling ---
        self.conv = nn.Conv2d(in_channels=3, out_channels=D, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn = nn.BatchNorm2d(D)                                                    
        self.pool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        # --- Selective Kernel ---
        self.sk = SKUnit(D, r=4)

        # --- Patch extraction ---
        self.unfold = nn.Unfold(kernel_size=P, stride=P) # extract non-overlapping P*P patches
        
        # --- Patch projection ---
        self.proj = nn.Linear(in_features=P*P*D, out_features=d) # map each patch to token embedding
    
        # add class token
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d))

    def forward(self, x):
        B = x.shape[0] # extract batch size

        # --- Initial convolution + downsampling ---
        x = self.conv(x)        # (B, D, H/2, W/2)
        x = self.bn(x)
        x = self.pool(x)        # (B, D, H/4, W/4)

        # --- SK unit ---
        x = self.sk(x)

        # --- Patch extraction ---
        x = self.unfold(x)      # (B, D*P*P, N)
        x = x.transpose(1,2)    # (B, N, D*P*P)

        # --- Patch projection ---
        x = self.proj(x)        # (B, N, d)

        # create class token
        cls = self.cls_token.expand(B, -1, -1)  # (B, 1, d)
        # prepend CLS token to patch tokens
        x = torch.cat([cls, x], dim=1)          #  (B, N+1, d)

        return x 

**Mixed Depthwise Convolution (MixConv) module** 

(Required for MCF module) 

The paper does not specify whether an equal or exponential channel partitioning method was used. Here, we use equal partitioning. This requires input channels to be divisible by the number of groups. The paper specifies kernel sizes (3, 5, 7).

In [7]:

class MixConv(nn.Module):
    """
    Mixed Depthwise Convolution (MixConv).

    Splits input channels into equal groups, applies depthwise convolution
    with a distinct kernel size to each group, and then concatenates the results.

    Args:
        in_channels (int): Number of input channels (C).
        kernel_sizes (list[int]): Kernel sizes per group (e.g. [3, 5, 7]).

    Input:
        x (Tensor): Shape (B, C, H, W)

    Output:
        Y (Tensor): shape (B, C, H, W)
    """

    def __init__(self, in_channels=512, kernel_sizes=(3, 5, 7)):
        super().__init__()

        # Store hyperparameters
        self.in_channels = in_channels
        self.kernel_sizes = list(kernel_sizes)
        num_groups = len(self.kernel_sizes)

        # --- Channel-wise partitioning ---
        # Split channels evenly across groups
        splits = [in_channels // num_groups] * num_groups
        self.splits = splits

        # --- Multi-scale depth-wise convolution ---
        self.convs = nn.ModuleList()
        for c, k in zip(self.splits, self.kernel_sizes):
            self.convs.append(
                nn.Conv2d(
                    in_channels=c,
                    out_channels=c,
                    kernel_size=k,
                    padding= (k-1) // 2,   
                    groups=c, # depth-wise convolution
                    bias=True
                )
            )

    def forward(self, x):

        # --- Partition input into groups ---
        x_groups = torch.split(x, self.splits, dim=1)

        # --- Apply multi-scale depth-wise convolution ---
        # (different kernel size for each group)
        y_groups = []
        for conv, xi in zip(self.convs, x_groups):
            y_groups.append(conv(xi))

        # --- Concatenate group outputs ---
        Y = torch.cat(y_groups, dim=1)

        return Y

**Mixed Convolutional Feed-forward (MCF)**
Implementation of the MCF described in the paper. 

In [ ]:
class MCF(nn.Module):
    """
    Mixed Convolutional Feed-forward module. First, split x into patch tokens and class tokens. 
    The patch tokens are then morphed to 2D, then a deep convolution is performed with kernel of 3. 
    Followed by a 1 x 1 convolution operation.
    Then a MixConv operation is applied which is followed by a 1x1 convolution operation.
    Finally, the feastures are reshaped from 2D to 1D 

    Args:
        

    Input:
        

    Output:
        
    """

    def __init__(self, dim, expansion=4):
        super().__init__()
        # e x d , expansion rate 
         dim2 = dim * expansion
        
        # 3x3 2D Conv 
        self.dwconv = nn.Conv2d(
            dim, 
            dim,
            kernel_size = 3, 
            padding = 1,
            groups = dim
        )

        ## 1x1 Conv 
        self.conv1 =  nn.Conv2d(
            dim,
            dim2,
            kernel_size=1
        ) 

        # MixConv 
        self.mixconv = MixConv(hidden_dim)

        self.conv2 = nn.Conv2d(
            dim2,
            dim,
            kernel_size=1
        )

        self.act = nn.GELU()
        

    def forward(self, x):
        B, N, C = x.shape

        cls_token = x[:, :1]
        x = x[:, 1:]

        H = W = int(N ** 0.5)

        x = x.reshape(B, H, W, C)
        x = x.permute(0, 3, 1, 2)

        x = self.dwconv(x)
        x = self.act(x)

        x = self.conv1(x)
        x = self.act(x)

        x = self.mixconv(x)
        x = self.act(x)

        x = self.conv2(x)

        x = x.flatten(2).transpose(1, 2)

        x = torch.cat([cls_token, x], dim=1)

       
        
        return x

**MSA Layer** 

In [ ]:
class MSA(nn.Module):
    def __init__(self, dim, heads=8):
        super().__init__()

        self.attn = nn.MultiheadAttention(
            dim,
            heads,
            batch_first=True
        )

    def forward(self, x):
        out, _ = self.attn(x, x, x)
        return out

**Encoder Layer**

In [ ]:
class EncoderLayer(nn.Module):
    """
    Encoder layer, comprised of a Multihead Self-Attention Module and a Mixed Convolutional Feed-forward module. 

    Args:
        

    Input:
        

    Output:
        
    """
    def __init__(self, dim, heads = 8):
        super().__init__()
            
        self.norm1 = nn.LayerNorm(dim)
        self.msa = MSA(dim, heads)

        self.norm2 = nn.LayerNorm(dim)
        self.mcf = MCF(dim)

    def forward(self, x):
        # Run MSA Layer 
        x = x + self.msa(self.norm1(x)) 

        # Run MCF Layer 
        x = x + self.mcf(self.norm2(x))
        return x 
        

**MFS**

In [ ]:
class MFS(nn.Module):
    def __init__(self, M):
        super().__init__()
        self.M = M

  def forward(self, attn_maps, tokens):
        B, L, N, _ = attn_maps.shape

        selected = []

        for l in range(L):
            cls_attn = attn_maps[:, l, 0]  # (B, N)

            topk = torch.topk(cls_attn, self.M, dim=-1).indices

            batch_selected = []
            for b in range(B):
                batch_selected.append(tokens[b, l, topk[b]])

            selected.append(torch.stack(batch_selected))

        # concat layers
        return torch.cat(selected, dim=1)

**HVCNet Model**

Following the architecture in figure 1 of the paper, we can develop the following model. 

In [ ]:
class HVC(nn.Module):
    def __init__(
        self,  
        dim = 384, 
        depth=10,
        M=32
    ):
        super().__init__()

        self.mit = MIT(d=dim)

        self.encode = 

    def forward(self, x):
        

## Implementation 1 - ...

In [1]:
#

## Implementation 2 - ...

In [2]:
#